# Part 8 (실습) — RAG: 우리 문서에서 찾아 답하기

> **이 노트북의 목표**
> 리리마켓 정책 문서를 청킹·임베딩해 벡터스토어에 저장하고, retriever로 의미 검색을 확인한 뒤, LCEL로 RAG 체인을 조립해 "문서 근거로 답하는" 봇을 완성함. 키워드 vs 의미 검색 차이도 비교함.
>
> **사용 모델**: 채팅 `gemini-3-flash`, 임베딩 `gemini-embedding-001` · **선행**: Part 0~7

## 0. 준비

In [ ]:
%pip install -q -U langchain langchain-google-genai langchain-text-splitters

In [ ]:
import os
from getpass import getpass
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Gemini API 키: ")
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
model = ChatGoogleGenerativeAI(model="gemini-3-flash", temperature=0)
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
print("준비 완료")

## 1. 문서 불러오기 & 청킹

긴 문서를 검색하기 좋은 조각으로 자름. `chunk_size`(조각 크기), `chunk_overlap`(경계 맥락 보존).

In [ ]:
policy = """
리리마켓 반품 및 교환 정책

1. 반품 기간: 상품 수령 후 7일 이내에 반품 신청이 가능합니다.
   단순 변심의 경우 왕복 배송비는 고객 부담입니다.

2. 교환: 동일 상품의 다른 사이즈/색상으로 교환은 14일 이내 가능합니다.
   재고가 없는 경우 환불로 처리됩니다.

3. 환불: 반품 상품 도착 확인 후 3영업일 이내에 결제 수단으로 환불됩니다.
   카드 취소는 카드사 사정에 따라 추가 영업일이 소요될 수 있습니다.

4. 반품 불가: 착용 흔적이 있거나 택을 제거한 상품, 세일 특가 상품은
   반품 및 교환이 제한됩니다.

5. 배송: 평일 오후 2시 이전 주문은 당일 출고되며, 도서산간은 1~2일 추가됩니다.
"""

from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
chunks = splitter.split_text(policy)

print(f"조각 {len(chunks)}개로 분할:")
for i, c in enumerate(chunks):
    print(f"  [{i}] {c.strip()[:50]}...")

## 2. 임베딩 — 텍스트를 의미의 좌표로

`GoogleGenerativeAIEmbeddings.embed_query()`가 텍스트를 숫자 벡터로 바꿈. 의미가 비슷하면 가까운 좌표가 됨.

In [ ]:
vec = embeddings.embed_query("반품은 어떻게 하나요?")
print("벡터 차원:", len(vec))
print("앞 5개 값:", [round(x, 4) for x in vec[:5]])

### 의미가 가까운 두 문장은 정말 가까울까?
"반품"과 "환불"은 글자가 다르지만 의미가 가까움. 코사인 유사도로 확인.

In [ ]:
import numpy as np
def cos_sim(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

q = embeddings.embed_query("환불 받고 싶어요")
near = embeddings.embed_query("반품 후 결제 수단으로 돌려받기")   # 의미 가까움
far  = embeddings.embed_query("오늘 점심 메뉴 추천해줘")          # 의미 멈

print(f"가까운 의미 유사도: {cos_sim(q, near):.3f}")
print(f"먼 의미   유사도: {cos_sim(q, far):.3f}")
print("→ 글자가 달라도 의미가 가까우면 유사도가 높음 (이것이 의미 검색의 힘)")

## 3. 벡터스토어 & retriever

조각들을 임베딩해 저장하고, 질문과 가까운 조각을 꺼내옴.

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

store = InMemoryVectorStore.from_texts(chunks, embedding=embeddings)
retriever = store.as_retriever(search_kwargs={"k": 2})   # 가까운 2개

docs = retriever.invoke("환불은 며칠 걸리나요?")
print("검색된 조각:")
for d in docs:
    print(" •", d.page_content.strip()[:60])

> 📌 "환불 며칠"이라고 물었더니 환불 기간이 적힌 조각이 검색됨. 키워드가 정확히 일치하지 않아도 의미로 찾아옴.

## 4. 키워드 검색 vs 의미 검색

글자가 안 겹치는 질문으로 차이를 봄. "돈 돌려받기"엔 '환불'이라는 단어가 없지만, 의미 검색은 환불 조항을 찾아옴.

In [ ]:
# 단순 키워드 매칭 흉내 — 질문의 단어가 조각에 그대로 있는지
def keyword_search(query, chunks):
    words = query.split()
    return [c for c in chunks if any(w in c for w in words)]

query = "돈 언제 돌려받아요?"
print("[키워드 검색]", "찾음" if keyword_search(query, chunks) else "못 찾음 (‘환불’이란 단어가 질문에 없음)")

print("[의미 검색]")
for d in retriever.invoke(query):
    print("  •", d.page_content.strip()[:60])

## 5. RAG 체인 조립

Part 3의 `RunnablePassthrough`로 "질문 그대로 + 동시에 문서 검색"을 프롬프트에 넣음.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    "다음 문서를 근거로만 답해줘. 문서에 없으면 '문서에서 찾을 수 없습니다'라고 해.\n\n"
    "[문서]\n{context}\n\n[질문]\n{question}"
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

print(rag_chain.invoke("반품은 며칠 안에 신청해야 하나요?"))

In [ ]:
# 여러 질문으로 확인
for q in ["환불은 얼마나 걸려요?", "세일 상품도 반품 되나요?", "주문하면 언제 출고돼요?"]:
    print(f"Q: {q}")
    print(f"A: {rag_chain.invoke(q)}\n")

### 환각 방지 확인 — 문서에 없는 질문
"문서 근거로만, 없으면 모른다고" 지시 덕분에 지어내지 않음.

In [ ]:
print(rag_chain.invoke("리리마켓 멤버십 등급은 몇 단계야?"))
# → 정책 문서에 없는 내용이므로 "문서에서 찾을 수 없습니다" 류로 답해야 함

## ✏️ 미니 실습

**과제**: 위 정책 문서에 "6. 포장: 모든 상품은 친환경 종이 포장으로 발송됩니다." 한 줄을 추가하고, 다시 청킹→저장→RAG 체인을 만들어 "포장은 어떻게 와요?"라고 물어보기.

In [ ]:
# TODO: 직접 작성
# policy2 = policy + "\n6. 포장: ..."
# chunks2 = splitter.split_text(policy2)
# store2 = InMemoryVectorStore.from_texts(chunks2, embedding=embeddings)
# ... retriever2, rag_chain2 만들어 호출

<details><summary>👉 정답</summary>

```python
policy2 = policy + "\n6. 포장: 모든 상품은 친환경 종이 포장으로 발송됩니다."
chunks2 = splitter.split_text(policy2)
store2 = InMemoryVectorStore.from_texts(chunks2, embedding=embeddings)
retriever2 = store2.as_retriever(search_kwargs={"k": 2})
rag_chain2 = (
    {"context": retriever2 | format_docs, "question": RunnablePassthrough()}
    | prompt | model | StrOutputParser()
)
print(rag_chain2.invoke("포장은 어떻게 와요?"))
```
</details>

## 정리

| 절 | 단계 | 핵심 |
|---|---|---|
| 1 | 청킹 | `RecursiveCharacterTextSplitter` |
| 2 | 임베딩 | 의미→좌표, 코사인 유사도 |
| 3 | 벡터스토어/retriever | `InMemoryVectorStore`, `as_retriever(k=)` |
| 4 | 키워드 vs 의미 | 글자 달라도 의미로 검색 |
| 5 | RAG 체인 | Passthrough + retriever를 프롬프트에 |

### 3줄 요약
1. RAG는 청킹→임베딩→저장→검색→근거 기반 생성의 흐름임.
2. 임베딩 덕분에 글자가 달라도 의미로 문서를 찾음.
3. RAG 체인은 초급 LCEL(Passthrough·파이프·파서)에 retriever를 끼운 것임.

### ▶ 다음 (Part 9)
드디어 `create_agent`로 진짜 에이전트를 제대로 만듦. 도구(7)와 RAG(8)를 쥔 에이전트가 LangGraph 위에서 스스로 일함.